In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("all_month.csv")

print("Dataset shape:", df.shape)

display(df.head())
display(df.tail())

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nSummary statistics:")
display(df.describe())

In [ ]:
# Missing-value report
missing_report = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage":
        (df.isnull().sum() / len(df) * 100).round(2)
})

display(
    missing_report.sort_values(
        "missing_percentage",
        ascending=False
    )
)

print("Duplicate rows:", df.duplicated().sum())
print("Duplicate IDs:", df["id"].duplicated().sum())

categorical_columns = [
    "magType",
    "type",
    "status",
    "locationSource",
    "magSource"
]

for column in categorical_columns:
    print(f"\n{column}")
    print(df[column].value_counts(dropna=False))
    
    
range_columns = [
    "latitude",
    "longitude",
    "depth",
    "mag",
    "nst",
    "gap",
    "dmin",
    "rms"
]

for column in range_columns:
    print(
        column,
        "Min:", df[column].min(),
        "Max:", df[column].max()
    )

In [ ]:
def clean_earthquake_data(df):
    df_clean = df.copy()

    df_clean.columns = (
        df_clean.columns
        .str.strip()
        .str.replace(
            r"([a-z0-9])([A-Z])",
            r"\1_\2",
            regex=True
        )
        .str.lower()
    )

    text_columns = df_clean.select_dtypes(
        include=["object", "string"]
    ).columns

    for column in text_columns:
        df_clean[column] = (
            df_clean[column]
            .astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )

    categorical_columns = [
        "mag_type",
        "type",
        "status",
        "location_source",
        "mag_source"
    ]

    for column in categorical_columns:
        if column in df_clean.columns:
            df_clean[column] = (
                df_clean[column]
                .str.lower()
                .str.strip()
            )

    numeric_columns = [
        "latitude",
        "longitude",
        "depth",
        "mag",
        "nst",
        "gap",
        "dmin",
        "rms",
        "horizontal_error",
        "depth_error",
        "mag_error",
        "mag_nst"
    ]

    for column in numeric_columns:
        if column in df_clean.columns:
            df_clean[column] = pd.to_numeric(
                df_clean[column],
                errors="coerce"
            )

    df_clean["time"] = pd.to_datetime(
        df_clean["time"],
        errors="coerce",
        utc=True
    )

    df_clean["updated"] = pd.to_datetime(
        df_clean["updated"],
        errors="coerce",
        utc=True
    )

    duplicate_rows_before = df_clean.duplicated().sum()
    duplicate_ids_before = df_clean["id"].duplicated().sum()

    df_clean = df_clean.drop_duplicates()
    df_clean = df_clean.drop_duplicates(
        subset="id",
        keep="last"
    )

    invalid_latitude = (
        df_clean["latitude"].notna()
        & ~df_clean["latitude"].between(-90, 90)
    )

    invalid_longitude = (
        df_clean["longitude"].notna()
        & ~df_clean["longitude"].between(-180, 180)
    )

    invalid_latitude_count = invalid_latitude.sum()
    invalid_longitude_count = invalid_longitude.sum()

    df_clean = df_clean[
        ~invalid_latitude
    ].copy()

    invalid_longitude = (
        df_clean["longitude"].notna()
        & ~df_clean["longitude"].between(-180, 180)
    )

    df_clean = df_clean[
        ~invalid_longitude
    ].copy()

    invalid_gap = (
        df_clean["gap"].notna()
        & ~df_clean["gap"].between(0, 360)
    )

    invalid_gap_count = invalid_gap.sum()

    df_clean.loc[
        invalid_gap,
        "gap"
    ] = np.nan

    non_negative_columns = [
        "nst",
        "dmin",
        "rms",
        "horizontal_error",
        "depth_error",
        "mag_error",
        "mag_nst"
    ]

    invalid_negative_counts = {}

    for column in non_negative_columns:
        if column in df_clean.columns:
            invalid_negative = (
                df_clean[column].notna()
                & (df_clean[column] < 0)
            )

            invalid_negative_counts[column] = invalid_negative.sum()

            df_clean.loc[
                invalid_negative,
                column
            ] = np.nan

    missing_report = pd.DataFrame({
        "missing_count": df_clean.isnull().sum(),
        "missing_percentage": (
            df_clean.isnull().sum()
            / len(df_clean)
            * 100
        ).round(2)
    }).sort_values(
        "missing_percentage",
        ascending=False
    )

    outlier_columns = ["mag", "depth", "nst", "gap", "dmin", "rms"]
    outlier_results = []

    for column in outlier_columns:
        # Filter for mag >= 3.0 dynamically without touching the main DataFrame
        data = df_clean[df_clean["mag"] >= 3.0][column].dropna()

        if data.empty:
            continue

        # Calculate percentiles based only on the filtered population
        lower_bound = data.quantile(0.01)
        upper_bound = data.quantile(0.99)

        # Prevent variables that cannot physically be negative
        if column in ["nst", "gap", "dmin", "rms"]:
            lower_bound = max(0, lower_bound)

        # Count values outside the boundaries
        outlier_count = ((data < lower_bound) | (data > upper_bound)).sum()

        outlier_results.append({
            "variable": column,
            "lower_bound": lower_bound,
            "upper_bound": upper_bound,
            "potential_outliers": outlier_count
        })

    
    outlier_summary = pd.DataFrame(outlier_results)

    
    df_clean["event_date"] = df_clean["time"].dt.date
    df_clean["event_year"] = df_clean["time"].dt.year
    df_clean["event_month"] = df_clean["time"].dt.month
    df_clean["event_day"] = df_clean["time"].dt.day
    df_clean["event_hour"] = df_clean["time"].dt.hour

    df_clean = df_clean.reset_index(drop=True)

    validation_summary = {
        "original_rows": len(df),
        "cleaned_rows": len(df_clean),
        "rows_removed": len(df) - len(df_clean),
        "duplicate_rows_before": int(duplicate_rows_before),
        "duplicate_ids_before": int(duplicate_ids_before),
        "duplicate_rows_after": int(df_clean.duplicated().sum()),
        "duplicate_ids_after": int(df_clean["id"].duplicated().sum()),
        "invalid_latitude_found": int(invalid_latitude_count),
        "invalid_longitude_found": int(invalid_longitude_count),
        "invalid_gap_found": int(invalid_gap_count),
        "invalid_negative_values": invalid_negative_counts
    }

    
    return (
        df_clean,
        missing_report,
        outlier_summary,
        validation_summary
    )



In [ ]:
df_clean, missing_report, outlier_summary, validation_summary = (
    clean_earthquake_data(df)
)

display(pd.DataFrame([validation_summary]))
display(missing_report)
display(outlier_summary)

df_clean.to_csv(
    "all_month_cleaned.csv",
    index=False
)

In [ ]:
eda_columns = [
    "mag",
    "depth",
    "nst",
    "gap",
    "dmin",
    "rms"
]

display(
    df_clean[eda_columns].describe()
)

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.hist(
    df_clean["mag"].dropna(),
    bins=30
)

plt.xlabel("Magnitude")
plt.ylabel("Number of Events")
plt.title("Distribution of Earthquake Magnitude")

plt.show()

plt.figure(figsize=(8, 5))

plt.hist(
    df_clean["depth"].dropna(),
    bins=30
)

plt.xlabel("Depth")
plt.ylabel("Number of Events")
plt.title("Distribution of Earthquake Depth")

plt.show()

plt.figure(figsize=(8, 4))

plt.boxplot(
    df_clean["mag"].dropna(),
    vert=False
)

plt.xlabel("Magnitude")
plt.title("Boxplot of Earthquake Magnitudes")
plt.show()

event_counts = (
    df_clean["type"]
    .value_counts()
)

event_counts.plot(
    kind="bar",
    figsize=(9, 5)
)

plt.xlabel("Event Type")
plt.ylabel("Number of Events")
plt.title("Distribution of Seismic Event Types")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.xlabel("Event Type")
plt.ylabel("Number of Events")
plt.title("Distribution of Seismic Event Types")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.xlabel("Event Type")
plt.ylabel("Number of Events")
plt.title("Distribution of Seismic Event Types")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    df_clean["depth"],
    df_clean["mag"],
    alpha=0.4
)

plt.xlabel("Depth")
plt.ylabel("Magnitude")
plt.title("Earthquake Magnitude vs Depth")

plt.show()

plt.figure(figsize=(8, 5))

plt.scatter(
    df_clean["nst"],
    df_clean["mag"],
    alpha=0.4
)

plt.xlabel("Number of Stations")
plt.ylabel("Magnitude")
plt.title("Magnitude vs Number of Stations")

plt.show()

plt.figure(figsize=(8, 5))

plt.scatter(
    df_clean["nst"],
    df_clean["mag"],
    alpha=0.4
)

plt.xlabel("Number of Stations")
plt.ylabel("Magnitude")
plt.title("Magnitude vs Number of Stations")

plt.show()

correlation_columns = [
    "latitude",
    "longitude",
    "depth",
    "mag",
    "nst",
    "gap",
    "dmin",
    "rms",
    "horizontal_error",
    "depth_error",
    "mag_error",
    "mag_nst"
]

correlation_matrix = (
    df_clean[correlation_columns]
    .corr()
)

display(correlation_matrix)

import seaborn as sns

plt.figure(figsize=(11, 8))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)

plt.title(
    "Correlation Matrix of Numerical Variables"
)

plt.tight_layout()
plt.show()

df_clean["date"] = (
    df_clean["time"]
    .dt.date
)

events_per_day = (
    df_clean
    .groupby("date")
    .size()
)

plt.figure(figsize=(12, 5))

events_per_day.plot()

plt.xlabel("Date")
plt.ylabel("Number of Events")
plt.title("Number of Seismic Events Per Day")

plt.tight_layout()
plt.show()

In [ ]:
# Select only columns that contain missing values
missing_to_plot = missing_report[
    missing_report["missing_count"] > 0
].sort_values("missing_percentage")

# Create a horizontal bar chart of missing-value percentages
plt.figure(figsize=(9, 5))

plt.barh(
    missing_to_plot.index,
    missing_to_plot["missing_percentage"]
)

# Add labels and a title
plt.xlabel("Missing Values (%)")
plt.ylabel("Feature")
plt.title("Missing-Value Percentage by Feature")

# Prevent labels from being cut off
plt.tight_layout()

# Display the graph
plt.show()

In [ ]:
# Count how many records use each magnitude measurement type
mag_type_counts = df_clean["mag_type"].value_counts(dropna=False)

# Create the bar chart
plt.figure(figsize=(9, 5))

mag_type_counts.plot(kind="bar")

# Add labels and title
plt.xlabel("Magnitude Type")
plt.ylabel("Number of Events")
plt.title("Distribution of Magnitude Measurement Types")

# Rotate labels so they are easier to read
plt.xticks(rotation=45, ha="right")

# Prevent labels from being cut off
plt.tight_layout()

# Display the graph
plt.show()

In [ ]:
# Count the number of records belonging to each event type
type_counts = df_clean["type"].value_counts()

# Only use event types that contain at least 20 observations
common_types = type_counts[
    type_counts >= 20
].index

# Create a list containing magnitude values for each common event type
magnitude_by_type = [
    df_clean.loc[
        df_clean["type"] == event_type,
        "mag"
    ].dropna()
    for event_type in common_types
]

# Create the boxplot
plt.figure(figsize=(9, 5))

plt.boxplot(
    magnitude_by_type,
    tick_labels=common_types
)

# Add labels and title
plt.xlabel("Event Type")
plt.ylabel("Magnitude")
plt.title("Magnitude Distribution by Common Event Type")

# Rotate labels
plt.xticks(rotation=30, ha="right")

plt.tight_layout()
plt.show()

In [ ]:
# Remove records that are missing longitude, latitude or magnitude
spatial_data = df_clean.dropna(
    subset=[
        "longitude",
        "latitude",
        "mag"
    ]
)

# Create the figure
plt.figure(figsize=(12, 6))

# Plot each event using longitude and latitude
scatter = plt.scatter(
    spatial_data["longitude"],
    spatial_data["latitude"],
    c=spatial_data["mag"],
    cmap="viridis",
    alpha=0.55,
    s=18
)

# Add a colour scale showing magnitude
plt.colorbar(
    scatter,
    label="Magnitude"
)

# Add axis labels and title
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Geographic Distribution of Recorded Seismic Events")

# Use valid worldwide latitude/longitude limits
plt.xlim(-180, 180)
plt.ylim(-90, 90)

# Add a light grid
plt.grid(alpha=0.2)

plt.tight_layout()
plt.show()

Phase 2

In [ ]:
df_raw = df.copy()

print("RAW DATASET SUMMARY")
print("-" * 40)

print(f"Number of rows: {df_raw.shape[0]}")
print(f"Number of columns: {df_raw.shape[1]}")
print(f"Duplicate rows: {df_raw.duplicated().sum()}")

print("\nDATA TYPES")
print(df_raw.dtypes)

missing_summary = pd.DataFrame({
    "Missing Values": df_raw.isnull().sum(),
    "Missing Percentage (%)": (
        df_raw.isnull().sum() / len(df_raw) * 100
    ).round(2)
})

missing_summary = missing_summary[
    missing_summary["Missing Values"] > 0
].sort_values("Missing Percentage (%)", ascending=False)

print("\nMISSING VALUES")
display(missing_summary)

unique_summary = pd.DataFrame({
    "Unique Values": df_raw.nunique(dropna=True)
}).sort_values("Unique Values")

print("\nUNIQUE VALUES")
display(unique_summary)

In [ ]:
categorical_columns = df_raw.select_dtypes(
    include=["object", "string"]
).columns

print(f"Number of categorical columns: {len(categorical_columns)}")

for column in categorical_columns:
    print(f"\n{column}")
    print(df_raw[column].value_counts(dropna=False).head(10))

In [ ]:
df_clean = df_raw.copy()

rows_before = len(df_clean)

text_columns = df_clean.select_dtypes(
    include=["object", "string"]
).columns

for column in text_columns:
    df_clean[column] = df_clean[column].apply(
        lambda value: value.strip()
        if isinstance(value, str)
        else value
    )

df_clean[text_columns] = df_clean[text_columns].replace(
    r"^\s*$",
    np.nan,
    regex=True
)

duplicate_count = df_clean.duplicated().sum()

df_clean = df_clean.drop_duplicates().reset_index(drop=True)

rows_after = len(df_clean)

print("BASIC CLEANING COMPLETE")
print("-" * 40)
print(f"Rows before cleaning: {rows_before}")
print(f"Duplicate rows removed: {duplicate_count}")
print(f"Rows after cleaning: {rows_after}")

In [ ]:

cleaning_summary = pd.DataFrame({
    "Dataset": ["Raw dataset", "Cleaned dataset"],
    "Rows": [
        df_raw.shape[0],
        df_clean.shape[0]
    ],
    "Columns": [
        df_raw.shape[1],
        df_clean.shape[1]
    ],
    "Missing Values": [
        df_raw.isnull().sum().sum(),
        df_clean.isnull().sum().sum()
    ],
    "Duplicate Rows": [
        df_raw.duplicated().sum(),
        df_clean.duplicated().sum()
    ]
})

display(cleaning_summary)

In [ ]:
# ============================================================
# PHASE 2 - DATA TYPE STANDARDISATION
# ============================================================

# Store the data types before conversion
dtypes_before = df_clean.dtypes.astype(str)

# Convert timestamp columns into datetime format
datetime_columns = [
    "time",
    "updated"
]

for column in datetime_columns:
    if column in df_clean.columns:
        df_clean[column] = pd.to_datetime(
            df_clean[column],
            errors="coerce",
            utc=True
        )


# Ensure measurement columns are stored as numeric values
numeric_columns = [
    "latitude",
    "longitude",
    "depth",
    "mag",
    "nst",
    "gap",
    "dmin",
    "rms",
    "horizontalError",
    "depthError",
    "magError",
    "magNst"
]

for column in numeric_columns:
    if column in df_clean.columns:
        df_clean[column] = pd.to_numeric(
            df_clean[column],
            errors="coerce"
        )


# Compare data types before and after preprocessing
datatype_summary = pd.DataFrame({
    "Before": dtypes_before,
    "After": df_clean.dtypes.astype(str)
})

datatype_summary["Changed"] = (
    datatype_summary["Before"]
    != datatype_summary["After"]
)

display(datatype_summary)

In [ ]:
# ============================================================
# VALIDATE DATA TYPE CONVERSION
# ============================================================

missing_after_conversion = df_clean.isnull().sum()

print(
    "Total missing values after type conversion:",
    int(missing_after_conversion.sum())
)

display(
    missing_after_conversion[
        missing_after_conversion > 0
    ].sort_values(ascending=False)
)

In [ ]:
# ============================================================
# PHASE 2 - PHYSICAL AND LOGICAL VALUE VALIDATION
# ============================================================

validation_results = []

def check_range(column, minimum=None, maximum=None):
    """
    Check a numeric feature against its expected physical or
    logical range without immediately modifying the data.
    """

    if column not in df_clean.columns:
        return

    values = df_clean[column]

    invalid_mask = pd.Series(
        False,
        index=df_clean.index
    )

    if minimum is not None:
        invalid_mask |= (
            values.notna()
            & (values < minimum)
        )

    if maximum is not None:
        invalid_mask |= (
            values.notna()
            & (values > maximum)
        )

    validation_results.append({
        "Feature": column,
        "Minimum Allowed": minimum,
        "Maximum Allowed": maximum,
        "Invalid Values": int(invalid_mask.sum())
    })

# Geographic constraints
check_range(
    "latitude", 
    minimum=-90, 
    maximum=90
)

check_range(
    "longitude", 
    minimum=-180, 
    maximum=180
)

# Station gap is an angle
check_range(
    "gap", 
    minimum=0, 
    maximum=360
)

# Measurement fields that should not be negative
check_range("nst", minimum=0)
check_range("dmin", minimum=0)
check_range("rms", minimum=0)
check_range("horizontalError", minimum=0)
check_range("depthError", minimum=0)
check_range("magError", minimum=0)
check_range("magNst", minimum=0)

validation_summary = pd.DataFrame(
    validation_results
)

display(validation_summary)

In [ ]:
# ============================================================
# HANDLE INVALID VALUES
# ============================================================

# Replace invalid geographic values with NaN
if "latitude" in df_clean.columns:

    invalid_latitude = (
        df_clean["latitude"].notna()
        & ~df_clean["latitude"].between(-90, 90)
    )

    df_clean.loc[
        invalid_latitude,
        "latitude"
    ] = np.nan


if "longitude" in df_clean.columns:

    invalid_longitude = (
        df_clean["longitude"].notna()
        & ~df_clean["longitude"].between(-180, 180)
    )

    df_clean.loc[
        invalid_longitude,
        "longitude"
    ] = np.nan


# Replace invalid station-gap values with NaN
if "gap" in df_clean.columns:

    invalid_gap = (
        df_clean["gap"].notna()
        & ~df_clean["gap"].between(0, 360)
    )

    df_clean.loc[
        invalid_gap,
        "gap"
    ] = np.nan


# Measurement fields that must be non-negative
non_negative_columns = [
    "nst",
    "dmin",
    "rms",
    "horizontalError",
    "depthError",
    "magError",
    "magNst"
]

for column in non_negative_columns:

    if column in df_clean.columns:

        invalid_negative = (
            df_clean[column].notna()
            & (df_clean[column] < 0)
        )

        df_clean.loc[
            invalid_negative,
            column
        ] = np.nan


print("Physical and logical validation complete.")
print(
    "Total missing values after validation:",
    int(df_clean.isnull().sum().sum())
)

In [ ]:
# PHASE II - PREPARE DATA FOR LCS
df_lcs = pd.read_csv("all_month_cleaned.csv")

# Remove columns that identify the individual earthquake
# rather than describing its characteristics
columns_to_remove = [
    "id",
    "updated",
    "place",
    "event_date"
]

df_lcs = df_lcs.drop(
    columns=[col for col in columns_to_remove if col in df_lcs.columns]
)

# Convert categorical columns to numeric dummy variables
categorical_columns = df_lcs.select_dtypes(
    include=["object"]
).columns.tolist()

df_lcs = pd.get_dummies(
    df_lcs,
    columns=categorical_columns,
    dtype=int
)

# Make sure all remaining values are numeric
df_lcs = df_lcs.apply(pd.to_numeric, errors="coerce")

# Handle any remaining missing values
df_lcs = df_lcs.fillna(df_lcs.median(numeric_only=True))

print("LCS dataset shape:", df_lcs.shape)
print("\nData types:")
print(df_lcs.dtypes.value_counts())

df_lcs.head()

In [ ]:
# PHASE II - eLCS MODEL CONFIGURATION
from skeLCS.eLCS import eLCS

# Reproducible eLCS configuration
lcs_config = {
    "learning_iterations": 10000,
    "N": 1000,
    "p_spec": 0.5,
    "nu": 5,
    "chi": 0.8,
    "mu": 0.04,
    "theta_GA": 25,
    "theta_del": 20,
    "theta_sub": 20,
    "acc_sub": 0.99,
    "beta": 0.2,
    "delta": 0.1,
    "init_fit": 0.01,
    "fitness_reduction": 0.1,
    "do_correct_set_subsumption": False,
    "do_GA_subsumption": True,
    "selection_method": "tournament",
    "theta_sel": 0.5,
    "random_state": 42
}

model = eLCS(**lcs_config)

print("eLCS model configured successfully.")
print("Learning iterations:", model.learning_iterations)
print("Maximum population size:", model.N)
print("Random state:", model.random_state)